# Day 2 — CI/CD with GitHub Actions

---

Yesterday you built an image manually. Today we automate it. Every push to `main` will:

1. Install deps
2. Run tests
3. Build the Docker image
4. Push it to a registry (GitHub Container Registry — free with your repo)

That's **CI/CD** — Continuous Integration + Continuous Deployment. Every serious AI team runs this.


## 1. GitHub Actions in one paragraph

GitHub Actions runs YAML-defined **workflows** on GitHub's servers in response to events (push, PR, schedule). Each workflow has one or more **jobs** made up of **steps**. Every push triggers a fresh Linux VM that runs your steps.

**Free tier: 2,000 minutes/month for private repos, unlimited for public.** Plenty for personal projects.


## 2. The file location that matters

GitHub Actions looks for YAML files in **`.github/workflows/`**. Any file there is a workflow.


## 3. A minimal CI workflow — test on every push

Create `.github/workflows/ci.yml` at the **repo root** (not next to this notebook):

```yaml
name: CI

on:
  push:
    branches: [main]
  pull_request:
  workflow_dispatch:      # lets you re-run manually from the Actions tab

jobs:
  test:
    runs-on: ubuntu-latest
    defaults:
      run:
        working-directory: Section_09_Deployment_MLOps/Day_2_CI_CD_GitHub_Actions
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - run: pip install pytest
      - run: pytest -q
```

**Why `working-directory`?** This repo has many `Section_*/Day_*` folders. Without scoping the working dir, `pytest` would walk the whole tree and try to run tests inside every `.venv` too. Scoping keeps the run fast and deterministic.

Commit this file, push. Open the repo's **Actions** tab — you'll see the workflow queue, run, and go green.


## 4. Add Docker build + push

Extend the workflow to build the image and push it to GHCR (GitHub Container Registry). GHCR is free with every repo and doesn't need a separate account — the token is auto-provisioned.

```yaml
  build-and-push:
    needs: test
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write

    steps:
      - uses: actions/checkout@v4
      - uses: docker/setup-buildx-action@v3

      - uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      # GHCR requires lowercase repo names. This step lowercases it.
      - id: meta
        run: echo "repo=${GITHUB_REPOSITORY,,}" >> "$GITHUB_OUTPUT"

      - uses: docker/build-push-action@v6
        with:
          context: Section_09_Deployment_MLOps/Day_1_Docker_For_AI
          push: ${{ github.event_name != 'pull_request' }}
          tags: |
            ghcr.io/${{ steps.meta.outputs.repo }}:latest
            ghcr.io/${{ steps.meta.outputs.repo }}:${{ github.sha }}
          cache-from: type=gha
          cache-to: type=gha,mode=max
```

**Four things worth pointing out:**

- `secrets.GITHUB_TOKEN` — auto-generated per run. You don't create it. The `permissions.packages: write` block is what lets it push to GHCR.
- `${GITHUB_REPOSITORY,,}` — GHCR rejects capital letters in image names (`Python_01` → `python_01`). The bash `,,` operator lowercases it.
- `context:` points at the Day 1 folder where the actual `Dockerfile` lives — the build context isn't always the repo root.
- Tagging with **both** `:latest` and `:${{ github.sha }}` means every commit gets an immutable image you can roll back to, and `latest` always points at `main`.
- `push: ${{ github.event_name != 'pull_request' }}` — build on PRs (so you catch broken Dockerfiles), but only push from `main`.


## 4b. What's actually wired up in this repo

The full workflow lives at [`.github/workflows/ci.yml`](../../.github/workflows/ci.yml) — it's the two jobs from sections 3 and 4 combined:

```
test  ─────►  build-and-push
              │
              ├─ context: Section_09_Deployment_MLOps/Day_1_Docker_For_AI
              └─ tags:    ghcr.io/udaykumarkakani/python_01:{latest, <sha>}
```

Verify it's running:

1. **Actions tab** → https://github.com/UdayKumarKakani/Python_01/actions
2. **Packages page** (after the first green run) → the image appears under the repo → *Packages*.
3. Pull it locally to prove the round-trip:

```bash
docker pull ghcr.io/udaykumarkakani/python_01:latest
docker run --rm -p 8000:8000 ghcr.io/udaykumarkakani/python_01:latest
curl http://localhost:8000/            # {"status":"ok",...}
```

By default GHCR images are **private** — first pull will fail with `denied`. Fix it in the package settings: **Package settings → Change visibility → Public**, or add a `docker login ghcr.io` step with a PAT (`read:packages` scope) for private pulls.


## 5. Managing secrets

Never commit API keys. GitHub → Settings → Secrets and variables → Actions → *New repository secret*.

In your workflow, use them like:

```yaml
      - name: Some step that needs the key
        env:
          TOGETHER_API_KEY: ${{ secrets.TOGETHER_API_KEY }}
        run: python scripts/check_api.py
```

For freshers: put `TOGETHER_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` in Actions secrets. Never inline them.


## 6. Testing AI code — what do you actually test?

You can't run real LLM calls in every CI run (cost + flakiness). Instead:

- **Unit tests** for pure functions: chunkers, formatters, retrieval logic. No LLM call.
- **Mock the LLM** for higher-level tests. `respx` / `pytest-mock`.
- **A single "smoke" integration test** using a real (cheap) API call, run only on `main`.

Sample smoke test:


In [ ]:
# tests/test_smoke.py
import os
import pytest

from main import ask   # your RAG function


@pytest.mark.skipif(not os.getenv("TOGETHER_API_KEY"), reason="no API key")
def test_ask_returns_something():
    out = ask("What is 2 + 2?")
    assert isinstance(out, str)
    assert len(out) > 0


## 7. Deploy step (preview)

Tomorrow's class (Day 3) covers deployment platforms. Once you're deploying to Render / Fly / Railway, the pattern is:

- **Render / Railway** — auto-deploy on push (they watch your repo). No workflow needed.
- **Fly.io** — add a step `flyctl deploy` to your workflow after the image is built.
- **AWS / GCP** — usually add a `terraform apply` or `kubectl apply` step.


## Recap

- **GitHub Actions** = free CI/CD triggered by events on your repo.
- Workflows live in **`.github/workflows/*.yml`**.
- **Test → Build → Push** is the standard AI-app pipeline.
- **Never commit secrets.** Use `${{ secrets.NAME }}` and set them in repo Settings.
- **Cache layers with `type=gha`** — dramatically faster after the first run.
- Mock LLM calls in most tests; run one real "smoke" test on `main`.
- **Next class:** actually deploying to a cloud host.
